# Lesson 4 : Tools in Responses API

Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, etc), and you can use these useful tools in your agent.<br>
With ```AzureAIClient``` in Agent Framwork, there exist 3 types of tools as follows. :

- Tools in Responses API : As I have mentioned in Lesson 1, ```AzureAIClient``` uses Azure OpenAI Responses API. You can work with tools in Responses API - such as, ```web_search_preview``` etc. You can also use built-in tools hosted on individual client in other clients. (For example, you can use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) when ```AnthropicClient```.)
- Tools in Foundry : Microsoft Foundry provides tools and you can work with theses tools in Agent Framework. We'll see this example in the next Lesson 5.
- Tools in Agent Framework : The library of Agent Framework SDK also provides native tools. These tools are mostly handled as local functions in LLM invocation, and these are processed in local computers (in which Agent Framework SDK runs).

In this example, we will explore web search tool and MCP tool in Responses API with Agent Framework.

---

Important Note : Currently, there is a bug in ```AzureAIClient``` that causes a warning when :
- You set ```use_latest_version=True``` option and get an agent.
- Run agent with the same ```AzureAIClient```, 2 or more times.

Please ignore warning.

---

## Web Search tool in Responses API

Same as in Lesson 1, we create a ```AzureAIClient``` object as follows.

In [1]:
from dotenv import load_dotenv
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Now we create tool definition for web search tool in Responses API, and create an agent with this tool setting.

By running ```get_web_search_tool()``` in ```AzureAIClient```, ```WebSearchPreviewTool``` is internally created and returned. (You can also explicitly create ```WebSearchPreviewTool``` object by yourself.)

In [2]:
from agent_framework import Agent

web_search_tool = client.get_web_search_tool()
agent = Agent(
    name="WeatherAgentWithSearchTool",
    client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[web_search_tool])

Now we run the agent as follows.

As you see, this agent knows "what day is it today" or "what is the actual weather condition today", because web search tool is used internally.

In Lesson 1, the tool execution is run on your local environment. In this example, Azure OpenAI Responses API will handle this process on server, because OpenAI's built-in web search tool is used this time.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")  # "今日の大阪の天気と気温を教えてください。"
display(Markdown(result.text))

Today in **Osaka (Fri, Jan 16, 2026)**: **mostly sunny**.

- **Temperature:** around **15°C / 59°F** high, **~5°C / 41°F** low [Osaka-shi, Osaka, Japan Weather Forecast | AccuWeather](https://www.accuweather.com/en/jp/osaka-shi/225007/weather-forecast/225007)[10 Day Weather - Osaka, Osaka, Japan - The Weather Channel](https://weather.com/weather/tenday/l/Osaka+Osaka+Japan?placeId=441174f51a1951566e6b1d02bd724effab80d7359e5f241540d1aa46dfecc59f)[Osaka, Japan 14 day weather forecast - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka/ext)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)

## MCP tool in Responses API

Next we explore MCP tool in Responses API with Agent Framework.

Same as above example, ```get_mcp_tool()``` method in ```AzureAIClient``` creates and returns ```MCPTool``` object, which is tool definition for Azure OpenAI Responses API.  

> Note : As I have mentioned above, Agent Framework also provides native built-in MCP tools - such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```. These are processed as local functions in Agent Framework SDK, in accordance with the MCP protocol specifications.  
> These might be useful when MCP (or some part of MCP specification) is not supported in your client - e.g., some remote API models might not support MCP Stdio server's tool. (However, this incurs overhead in your computing environment.)

In this example, we create an agent to answer Microsoft technical questions.  
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
mcp_tool = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)
agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",  # "あなたは、Microsoft の製品やサービスに関する技術質問に答える Agent です。"
    tools=[mcp_tool],
)

Let's ask a technical question about Microsoft Azure.  
In this call, MCP tool calling (about Microsoft Learn document) is used in Azure OpenAI Responses API internally. (Use [tracing](./02_trace.ipynb) and see the internal steps.)

In [5]:
from IPython.display import Markdown, display

result = await agent.run("How to create an Azure storage account using Azure CLI ?")  # "Azure CLI を使って Azure Storage アカウントを作成する方法を教えてください。"
display(Markdown(result.text))

To create an Azure Storage account with the Azure CLI, you typically:

1) **Sign in and pick a subscription**
```bash
az login
az account set --subscription "<SUBSCRIPTION_ID_OR_NAME>"
```

2) **Create a resource group** (skip if you already have one)
```bash
az group create \
  --name <RG_NAME> \
  --location <LOCATION>
```

3) **Create the storage account**
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2
```

### Common options you may want
- **Block public blob access** (recommended in many orgs):
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2 \
  --allow-blob-public-access false
```

- **Require secure transfer (HTTPS only)**:
```bash
az storage account update \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --https-only true
```

4) **Verify**
```bash
az storage account show \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME>
```

### Notes / gotchas
- `<STORAGE_ACCOUNT_NAME>` must be **globally unique**, **3–24** chars, **lowercase letters and numbers only**.
- `<LOCATION>` examples: `eastus`, `westus2`, `westeurope`.

If you tell me your intended region, redundancy (LRS/ZRS/GRS), and whether this is for blobs only or general use, I can suggest the best `--sku`/security flags.